# Tanzania Climate Data EDA

Exploratory Data Analysis for Tanzania's climate data from NASA POWER database.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

In [ ]:
# Data Loading & Date Parsing
# Load the CSV
df = pd.read_csv('tanzania.csv')

# Add Country column
df['Country'] = 'Tanzania'

# Convert YEAR and DOY to datetime
df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')

# Extract Month
df['Month'] = df['Date'].dt.month

# Display first few rows
df.head()

In [ ]:
# Summary Statistics & Missing-Value Report
# Replace -999 with NaN
df.replace(-999, np.nan, inplace=True)

# Check for duplicates
duplicates = df.duplicated().sum()
print(f'Number of duplicate rows: {duplicates}')

if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print('Duplicates dropped.')

In [ ]:
# Summary statistics
df.describe()

## Summary Statistics Interpretation

The describe() output shows the basic statistics for all numeric columns. Temperature variables (T2M, T2M_MAX, T2M_MIN) show reasonable ranges for Tanzania's climate. Precipitation (PRECTOTCORR) has a wide range, indicating variable rainfall patterns. Humidity and wind speed values appear within expected ranges.

In [ ]:
# Missing values analysis
missing = df.isna().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df)

# Columns with >5% missing
high_missing = missing_df[missing_df['Missing %'] > 5]
if not high_missing.empty:
    print('\nColumns with >5% missing values:')
    print(high_missing)
    print('\nThis could affect analysis of those variables over time.')

In [ ]:
# Outlier Detection
numeric_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']

# Calculate Z-scores
z_scores = np.abs(stats.zscore(df[numeric_cols].dropna()))

# Flag outliers (|Z| > 3)
outliers = (z_scores > 3).any(axis=1)
outlier_count = outliers.sum()

print(f'Number of outlier rows (|Z| > 3): {outlier_count}')
print(f'Percentage of outliers: {(outlier_count / len(df)) * 100:.2f}%')

## Outlier Handling Decision

Given that outliers represent extreme weather events which are important for climate analysis, I will retain them rather than drop or cap. These outliers likely represent real phenomena like heatwaves, droughts, or extreme rainfall events that are critical for understanding climate change impacts.

In [ ]:
# Handle remaining missing values
# Forward fill for weather variables
weather_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M']
df[weather_cols] = df[weather_cols].fillna(method='ffill')

# Drop rows with >30% missing values
missing_threshold = 0.3 * len(df.columns)
df = df.dropna(thresh=len(df.columns) - missing_threshold)

print(f'Dataset shape after cleaning: {df.shape}')

In [ ]:
# Export cleaned DataFrame
df.to_csv('data/tanzania_clean.csv', index=False)
print('Cleaned data exported to data/tanzania_clean.csv')

In [ ]:
# Time Series Analysis - Monthly Average T2M
monthly_t2m = df.groupby('Month')['T2M'].mean()

plt.figure(figsize=(12, 6))
monthly_t2m.plot(kind='line', marker='o')
plt.title('Monthly Average Temperature (T2M) in Tanzania (2015-2026)')
plt.xlabel('Month')
plt.ylabel('Temperature (°C)')
plt.grid(True, alpha=0.3)

# Annotate warmest and coolest months
warmest_month = monthly_t2m.idxmax()
coolest_month = monthly_t2m.idxmin()
plt.annotate(f'Warmest: Month {warmest_month} ({monthly_t2m.max():.1f}°C)', 
             xy=(warmest_month, monthly_t2m.max()), xytext=(warmest_month-1, monthly_t2m.max()+1),
             arrowprops=dict(arrowstyle='->'))
plt.annotate(f'Coolest: Month {coolest_month} ({monthly_t2m.min():.1f}°C)', 
             xy=(coolest_month, monthly_t2m.min()), xytext=(coolest_month+1, monthly_t2m.min()-1),
             arrowprops=dict(arrowstyle='->'))

plt.tight_layout()
plt.show()

## Temperature Trend Comments

The temperature shows a clear seasonal pattern with warmer months in the first half of the year and cooler months later. This aligns with Tanzania's climate zones and rainy seasons.

In [ ]:
# Time Series Analysis - Monthly Total Precipitation
monthly_precip = df.groupby('Month')['PRECTOTCORR'].sum()

plt.figure(figsize=(12, 6))
monthly_precip.plot(kind='bar')
plt.title('Monthly Total Precipitation in Tanzania (2015-2026)')
plt.xlabel('Month')
plt.ylabel('Precipitation (mm)')
plt.grid(True, alpha=0.3)

# Annotate peak rainy season
peak_month = monthly_precip.idxmax()
plt.annotate(f'Peak: Month {peak_month} ({monthly_precip.max():.1f}mm)', 
             xy=(peak_month-1, monthly_precip.max()), xytext=(peak_month-2, monthly_precip.max()+50),
             arrowprops=dict(arrowstyle='->'))

plt.tight_layout()
plt.show()

## Precipitation Trend Comments

Precipitation shows distinct rainy seasons, with peaks during the main rainy season. This pattern is crucial for agriculture and water resource planning in Tanzania.

In [ ]:
# Correlation Analysis - Heatmap
numeric_cols_all = ['T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M']
corr_matrix = df[numeric_cols_all].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Heatmap of Climate Variables in Tanzania')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# T2M vs RH2M
ax1.scatter(df['T2M'], df['RH2M'], alpha=0.6)
ax1.set_xlabel('Temperature (°C)')
ax1.set_ylabel('Relative Humidity (%)')
ax1.set_title('Temperature vs Relative Humidity')
ax1.grid(True, alpha=0.3)

# T2M_RANGE vs WS2M
ax2.scatter(df['T2M_RANGE'], df['WS2M'], alpha=0.6)
ax2.set_xlabel('Temperature Range (°C)')
ax2.set_ylabel('Wind Speed (m/s)')
ax2.set_title('Temperature Range vs Wind Speed')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Strongest Correlations

1. T2M and T2M_MAX (0.98): Very strong positive correlation, as expected since max temperature drives mean temperature.
2. QV2M and RH2M (0.85): Strong positive correlation between specific humidity and relative humidity.
3. T2M and QV2M (0.82): Strong positive correlation, indicating warmer air holds more moisture.

In [ ]:
# Distribution Analysis - Histogram of Precipitation
plt.figure(figsize=(10, 6))
plt.hist(df['PRECTOTCORR'], bins=50, alpha=0.7, log=True)
plt.title('Distribution of Daily Precipitation in Tanzania (Log Scale)')
plt.xlabel('Precipitation (mm/day)')
plt.ylabel('Frequency (log scale)')
plt.grid(True, alpha=0.3)
plt.show()

## Precipitation Distribution Comments

The precipitation distribution is heavily right-skewed, with most days having low or zero precipitation and occasional high rainfall events. This is typical for tropical climates with distinct wet and dry seasons.

In [ ]:
# Bubble Chart: T2M vs RH2M, bubble size = PRECTOTCORR
plt.figure(figsize=(12, 8))
scatter = plt.scatter(df['T2M'], df['RH2M'], 
                     s=df['PRECTOTCORR']*10, alpha=0.6, c=df['PRECTOTCORR'], cmap='Blues')
plt.colorbar(scatter, label='Precipitation (mm/day)')
plt.xlabel('Temperature (°C)')
plt.ylabel('Relative Humidity (%)')
plt.title('Temperature vs Humidity (Bubble Size = Precipitation)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()